# CNN + Genetic Algorithm + Fuzzy Inference System Image Classification

This notebook builds a complete binary image classification pipeline using:

- a Convolutional Neural Network (CNN) for feature learning and prediction,
- a Genetic Algorithm (GA) with `pygad` for hyperparameter optimization, and
- a Fuzzy Inference System (FIS) with `scikit-fuzzy` for the final decision layer.

The dataset is expected to live in `../dataset` relative to this notebook and use filenames like `001_neg.jpg` or `002_pos.png`.

Label mapping used throughout the notebook:

- `neg` -> `0` (`Negative`)
- `pos` -> `1` (`Positive`)


## 1. Dependencies and Environment Setup

Install the required libraries with `uv`:

```powershell
uv add tensorflow scikit-learn opencv-python pygad scikit-fuzzy jupyter matplotlib pandas seaborn
```

If you want the notebook kernel to immediately see the new packages, restart the kernel after installation.


In [3]:
from __future__ import annotations

import gc
import json
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pygad
import seaborn as sns
import skfuzzy as fuzz
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from skfuzzy import control as ctrl

SEED = 42
IMAGE_SIZE = (512, 512)
CHANNELS = 3
DATASET_DIR = Path("../dataset")
ARTIFACTS_DIR = Path("../artifacts")
MODEL_SAVE_PATH = ARTIFACTS_DIR / "cnn_fis_classifier.keras"
HYPERPARAMETERS_PATH = ARTIFACTS_DIR / "best_hyperparameters.json"
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.style.use("seaborn-v0_8")
sns.set_palette("deep")
print(f"TensorFlow version: {tf.__version__}")
print(f"Dataset directory: {DATASET_DIR.resolve()}")


TensorFlow version: 2.21.0
Dataset directory: C:\College_Computer_Science\Semester 6\Soft Comp\soft-computing-final\dataset


## 2. Data Loading and Preprocessing

The helper functions below:

- scan the `dataset` folder for image files,
- extract labels from filenames using the `<number>_<label>` convention,
- resize every image to `512 x 512`,
- normalize pixel values to `[0, 1]`, and
- return both a metadata table and NumPy arrays ready for CNN training.


In [ ]:
LABEL_MAP = {"neg": 0, "pos": 1}
LABEL_NAMES = {0: "Negative", 1: "Positive"}


def list_image_files(dataset_dir: Path) -> List[Path]:
    return sorted(
        path
        for path in dataset_dir.glob("**/*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )


def parse_label_from_filename(path: Path) -> int:
    """Extract the trailing filename token and map it into a numeric class."""
    stem_parts = path.stem.lower().split("_")
    if len(stem_parts) < 2:
        raise ValueError(
            f"Filename '{path.name}' does not follow the '<number>_<label>' format."
        )

    raw_label = stem_parts[-1]
    if raw_label not in LABEL_MAP:
        raise ValueError(
            f"Filename '{path.name}' contains label '{raw_label}', expected one of {list(LABEL_MAP)}."
        )
    return LABEL_MAP[raw_label]


def preprocess_image(image_path: Path, image_size: Tuple[int, int] = IMAGE_SIZE) -> np.ndarray:
    """Load an image with OpenCV, convert it to RGB, resize it, and normalize it."""
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, image_size, interpolation=cv2.INTER_AREA)
    image = image.astype(np.float32) / 255.0
    return image


def load_dataset(dataset_dir: Path) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    records: List[Dict[str, object]] = []
    images: List[np.ndarray] = []
    labels: List[int] = []

    for image_path in list_image_files(dataset_dir):
        label = parse_label_from_filename(image_path)
        image_array = preprocess_image(image_path)

        records.append(
            {
                "path": str(image_path),
                "filename": image_path.name,
                "label": label,
                "label_name": LABEL_NAMES[label],
            }
        )
        images.append(image_array)
        labels.append(label)

    dataframe = pd.DataFrame(records)
    if not images:
        empty_images = np.empty((0, IMAGE_SIZE[0], IMAGE_SIZE[1], CHANNELS), dtype=np.float32)
        empty_labels = np.empty((0,), dtype=np.int32)
        return dataframe, empty_images, empty_labels

    return dataframe, np.stack(images).astype(np.float32), np.asarray(labels, dtype=np.int32)


def plot_sample_images(images: np.ndarray, labels: np.ndarray, num_samples: int = 8) -> None:
    if len(images) == 0:
        print("No images available to preview.")
        return

    num_samples = min(num_samples, len(images))
    columns = min(4, num_samples)
    rows = int(np.ceil(num_samples / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for axis, index in zip(axes, range(num_samples)):
        axis.imshow(images[index])
        axis.set_title(LABEL_NAMES[int(labels[index])])
        axis.axis("off")

    for axis in axes[num_samples:]:
        axis.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
metadata_df, X, y = load_dataset(DATASET_DIR)
DATA_READY = len(metadata_df) > 0

print(f"Images found: {len(metadata_df)}")
if DATA_READY:
    display(metadata_df.head())
    display(metadata_df["label_name"].value_counts().rename_axis("class").reset_index(name="count"))
    plot_sample_images(X, y)
else:
    print(
        "The dataset folder is currently empty. The remaining cells are written and ready, "
        "but training will be skipped until images are added to ../dataset."
    )


## 3. Train/Validation/Test Split and TensorFlow Pipelines

We split the normalized image tensors into training, validation, and test sets. The validation split is used by the GA fitness function, while the test split is reserved for the final CNN + FIS evaluation.


In [ ]:
def ensure_training_ready(labels: np.ndarray) -> None:
    if len(labels) == 0:
        raise RuntimeError("No images were loaded. Add data to ../dataset and rerun the notebook.")
    if len(np.unique(labels)) < 2:
        raise RuntimeError("At least two classes are required to train the CNN.")
    if len(labels) < 10:
        raise RuntimeError(
            "The dataset is very small. Add more images so stratified train/validation/test splits are reliable."
        )


def make_splits(
    X: np.ndarray,
    y: np.ndarray,
    test_size: float = 0.2,
    val_size: float = 0.2,
    random_state: int = SEED,
) -> Dict[str, np.ndarray]:
    ensure_training_ready(y)

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )

    adjusted_val_size = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=adjusted_val_size,
        stratify=y_train_val,
        random_state=random_state,
    )

    return {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
    }


def make_tf_dataset(features: np.ndarray, labels: np.ndarray, batch_size: int, training: bool = False) -> tf.data.Dataset:
    dataset = tf.data.Dataset.from_tensor_slices((features, labels))
    if training:
        dataset = dataset.shuffle(buffer_size=len(features), seed=SEED, reshuffle_each_iteration=True)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


splits = None
if DATA_READY:
    try:
        splits = make_splits(X, y)
        for key, value in splits.items():
            print(f"{key}: {value.shape}")
    except RuntimeError as error:
        print(error)
else:
    print("Skipping split creation until dataset images are added.")


## 4. Dynamic CNN Architecture

The model-building function below accepts tunable hyperparameters so the GA can search over different CNN configurations.


In [ ]:
@dataclass(frozen=True)
class CNNHyperParameters:
    learning_rate: float
    filters_1: int
    filters_2: int
    dense_units: int
    dropout_rate: float
    batch_size: int


def build_cnn_model(
    input_shape: Tuple[int, int, int],
    hyperparameters: CNNHyperParameters,
) -> tf.keras.Model:
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=input_shape),
            tf.keras.layers.Conv2D(hyperparameters.filters_1, (3, 3), activation="relu", padding="same"),
            tf.keras.layers.MaxPooling2D((2, 2)),
            tf.keras.layers.Conv2D(hyperparameters.filters_2, (3, 3), activation="relu", padding="same"),
            tf.keras.layers.MaxPooling2D((2, 2)),
            tf.keras.layers.Conv2D(hyperparameters.filters_2, (3, 3), activation="relu", padding="same"),
            tf.keras.layers.GlobalAveragePooling2D(),
            tf.keras.layers.Dense(hyperparameters.dense_units, activation="relu"),
            tf.keras.layers.Dropout(hyperparameters.dropout_rate),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ]
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hyperparameters.learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


example_hparams = CNNHyperParameters(
    learning_rate=1e-3,
    filters_1=32,
    filters_2=64,
    dense_units=128,
    dropout_rate=0.3,
    batch_size=32,
)

build_cnn_model((IMAGE_SIZE[0], IMAGE_SIZE[1], CHANNELS), example_hparams).summary()


## 5. Hyperparameter Optimization with a Genetic Algorithm

The GA encodes a CNN configuration as a chromosome. The fitness function:

- decodes the chromosome into CNN hyperparameters,
- builds a fresh CNN,
- trains it for **exactly 5 epochs** on the training split, and
- returns validation accuracy as the fitness score.

This is intentionally expensive because every candidate solution trains a real network, so start with a small population and a few generations when experimenting.


In [ ]:
GENE_SPACE = [
    [1e-4, 3e-4, 1e-3],
    [16, 32, 48],
    [32, 64, 96],
    [64, 128, 256],
    [0.2, 0.3, 0.4, 0.5],
    [16, 32],
]

GENE_NAMES = [
    "learning_rate",
    "filters_1",
    "filters_2",
    "dense_units",
    "dropout_rate",
    "batch_size",
]


def decode_solution(solution: np.ndarray) -> CNNHyperParameters:
    values = {
        gene_name: gene_value
        for gene_name, gene_value in zip(GENE_NAMES, solution)
    }
    return CNNHyperParameters(
        learning_rate=float(values["learning_rate"]),
        filters_1=int(values["filters_1"]),
        filters_2=int(values["filters_2"]),
        dense_units=int(values["dense_units"]),
        dropout_rate=float(values["dropout_rate"]),
        batch_size=int(values["batch_size"]),
    )


def make_fitness_function(splits: Dict[str, np.ndarray]):
    def fitness_func(ga_instance: pygad.GA, solution: np.ndarray, solution_idx: int) -> float:
        """Train a candidate CNN for exactly 5 epochs and return validation accuracy."""
        hyperparameters = decode_solution(solution)
        tf.keras.backend.clear_session()
        gc.collect()

        model = build_cnn_model(
            (IMAGE_SIZE[0], IMAGE_SIZE[1], CHANNELS),
            hyperparameters,
        )

        train_dataset = make_tf_dataset(
            splits["X_train"],
            splits["y_train"],
            batch_size=hyperparameters.batch_size,
            training=True,
        )
        val_dataset = make_tf_dataset(
            splits["X_val"],
            splits["y_val"],
            batch_size=hyperparameters.batch_size,
            training=False,
        )

        history = model.fit(
            train_dataset,
            validation_data=val_dataset,
            epochs=5,
            verbose=0,
        )

        validation_accuracy = float(history.history["val_accuracy"][-1])
        return validation_accuracy

    return fitness_func


def run_ga_search(splits: Dict[str, np.ndarray]) -> Tuple[pygad.GA, CNNHyperParameters]:
    ga_instance = pygad.GA(
        num_generations=3,
        num_parents_mating=2,
        sol_per_pop=4,
        num_genes=len(GENE_SPACE),
        gene_space=GENE_SPACE,
        fitness_func=make_fitness_function(splits),
        parent_selection_type="sss",
        keep_parents=1,
        crossover_type="single_point",
        mutation_type="random",
        mutation_percent_genes=30,
        random_seed=SEED,
        save_solutions=True,
        suppress_warnings=True,
    )
    ga_instance.run()
    best_solution, best_solution_fitness, _ = ga_instance.best_solution()
    best_hyperparameters = decode_solution(best_solution)
    print(f"Best validation accuracy from GA: {best_solution_fitness:.4f}")
    print(best_hyperparameters)
    return ga_instance, best_hyperparameters


In [ ]:
ga_instance = None
best_hyperparameters = None

if splits is not None:
    ga_instance, best_hyperparameters = run_ga_search(splits)
else:
    print("Skipping GA because the dataset is not ready for training yet.")


## 6. Final CNN Training with the Best Hyperparameters

After the GA finishes, we rebuild the CNN using the best chromosome and train it again on the training data. The validation split is still monitored during training so we can inspect learning behavior.

At the end of training, the notebook also saves:

- the trained Keras model to `../artifacts/cnn_fis_classifier.keras`, and
- the selected hyperparameters to `../artifacts/best_hyperparameters.json`.


In [ ]:
final_model = None
final_history = None

if splits is not None:
    if best_hyperparameters is None:
        best_hyperparameters = example_hparams
        print("GA was skipped, so the example hyperparameters will be used for final training.")

    tf.keras.backend.clear_session()
    final_model = build_cnn_model(
        (IMAGE_SIZE[0], IMAGE_SIZE[1], CHANNELS),
        best_hyperparameters,
    )

    train_dataset = make_tf_dataset(
        splits["X_train"],
        splits["y_train"],
        batch_size=best_hyperparameters.batch_size,
        training=True,
    )
    val_dataset = make_tf_dataset(
        splits["X_val"],
        splits["y_val"],
        batch_size=best_hyperparameters.batch_size,
        training=False,
    )
    test_dataset = make_tf_dataset(
        splits["X_test"],
        splits["y_test"],
        batch_size=best_hyperparameters.batch_size,
        training=False,
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
        )
    ]

    final_history = final_model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=12,
        verbose=1,
        callbacks=callbacks,
    )

    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    final_model.save(MODEL_SAVE_PATH)
    with HYPERPARAMETERS_PATH.open("w", encoding="utf-8") as file:
        json.dump(best_hyperparameters.__dict__, file, indent=2)
    print(f"Saved trained model to: {MODEL_SAVE_PATH.resolve()}")
    print(f"Saved best hyperparameters to: {HYPERPARAMETERS_PATH.resolve()}")

    test_loss, test_accuracy = final_model.evaluate(test_dataset, verbose=0)
    print(f"Final test loss: {test_loss:.4f}")
    print(f"Final test accuracy: {test_accuracy:.4f}")
else:
    print("Skipping final training because the dataset is not ready.")


In [ ]:
if final_history is not None:
    history_df = pd.DataFrame(final_history.history)
    figure, axes = plt.subplots(1, 2, figsize=(14, 5))

    history_df[["loss", "val_loss"]].plot(ax=axes[0], title="Loss Curve")
    history_df[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy Curve")

    axes[0].set_xlabel("Epoch")
    axes[1].set_xlabel("Epoch")
    plt.tight_layout()
    plt.show()
else:
    print("No final training history to visualize yet.")


## 7. Fuzzy Inference System (FIS) for the Final Decision

The CNN outputs a probability for the positive class. We convert that into two confidence inputs:

- `positive_confidence = P(Positive)`
- `negative_confidence = 1 - P(Positive)`

A fuzzy system then maps those confidence values into one of three decision categories:

- `Negative`
- `Other/Uncertain`
- `Positive`

This is useful when probabilities are ambiguous, because the FIS can explicitly represent uncertainty instead of forcing every sample into a hard binary class.


In [ ]:
def build_fuzzy_system() -> ctrl.ControlSystemSimulation:
    positive_confidence = ctrl.Antecedent(np.linspace(0, 1, 101), "positive_confidence")
    negative_confidence = ctrl.Antecedent(np.linspace(0, 1, 101), "negative_confidence")
    final_decision = ctrl.Consequent(np.linspace(0, 100, 101), "final_decision")

    for variable in (positive_confidence, negative_confidence):
        variable["low"] = fuzz.trimf(variable.universe, [0.0, 0.0, 0.45])
        variable["medium"] = fuzz.trimf(variable.universe, [0.25, 0.5, 0.75])
        variable["high"] = fuzz.trimf(variable.universe, [0.55, 1.0, 1.0])

    final_decision["negative"] = fuzz.trimf(final_decision.universe, [0, 0, 35])
    final_decision["uncertain"] = fuzz.trimf(final_decision.universe, [25, 50, 75])
    final_decision["positive"] = fuzz.trimf(final_decision.universe, [65, 100, 100])

    rules = [
        ctrl.Rule(positive_confidence["high"] & negative_confidence["low"], final_decision["positive"]),
        ctrl.Rule(positive_confidence["medium"] & negative_confidence["low"], final_decision["positive"]),
        ctrl.Rule(negative_confidence["high"] & positive_confidence["low"], final_decision["negative"]),
        ctrl.Rule(negative_confidence["medium"] & positive_confidence["low"], final_decision["negative"]),
        ctrl.Rule(positive_confidence["medium"] & negative_confidence["medium"], final_decision["uncertain"]),
        ctrl.Rule(positive_confidence["high"] & negative_confidence["high"], final_decision["uncertain"]),
        ctrl.Rule(positive_confidence["low"] & negative_confidence["low"], final_decision["uncertain"]),
    ]

    control_system = ctrl.ControlSystem(rules)
    return ctrl.ControlSystemSimulation(control_system)


def fuzzy_label_from_probabilities(
    positive_probability: float,
    negative_probability: float | None = None,
) -> Tuple[str, float]:
    if negative_probability is None:
        negative_probability = 1.0 - positive_probability

    fuzzy_system = build_fuzzy_system()
    fuzzy_system.input["positive_confidence"] = float(positive_probability)
    fuzzy_system.input["negative_confidence"] = float(negative_probability)
    fuzzy_system.compute()

    decision_score = float(fuzzy_system.output["final_decision"])
    if decision_score >= 65:
        label = "Positive"
    elif decision_score <= 35:
        label = "Negative"
    else:
        label = "Other/Uncertain"

    return label, decision_score


In [ ]:
results_df = None

if final_model is not None and splits is not None:
    positive_probabilities = final_model.predict(test_dataset, verbose=0).reshape(-1)
    negative_probabilities = 1.0 - positive_probabilities
    cnn_predictions = (positive_probabilities >= 0.5).astype(int)

    fuzzy_outputs = [
        fuzzy_label_from_probabilities(pos_prob, neg_prob)
        for pos_prob, neg_prob in zip(positive_probabilities, negative_probabilities)
    ]

    results_df = pd.DataFrame(
        {
            "true_label": splits["y_test"],
            "true_label_name": [LABEL_NAMES[int(label)] for label in splits["y_test"]],
            "positive_probability": positive_probabilities,
            "negative_probability": negative_probabilities,
            "cnn_prediction": cnn_predictions,
            "cnn_prediction_name": [LABEL_NAMES[int(label)] for label in cnn_predictions],
            "fis_label": [label for label, _ in fuzzy_outputs],
            "fis_score": [score for _, score in fuzzy_outputs],
        }
    )

    display(results_df.head())

    print("CNN classification report:")
    print(classification_report(splits["y_test"], cnn_predictions, target_names=["Negative", "Positive"]))

    confusion = confusion_matrix(splits["y_test"], cnn_predictions)
    sns.heatmap(
        confusion,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Negative", "Positive"],
        yticklabels=["Negative", "Positive"],
    )
    plt.title("CNN Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    print("FIS decision counts:")
    display(results_df["fis_label"].value_counts().rename_axis("fis_label").reset_index(name="count"))
else:
    print("Skipping FIS evaluation because the final CNN model is not available yet.")


## 8. Inference Helper for New Images

Use the helper below to score any folder of images after the final CNN has been trained. The fuzzy layer will return `Positive`, `Negative`, or `Other/Uncertain` for each image.


In [ ]:
def predict_folder_with_fis(folder_path: str | Path, model: tf.keras.Model) -> pd.DataFrame:
    folder = Path(folder_path)
    image_paths = [path for path in list_image_files(folder)]
    if not image_paths:
        raise ValueError(f"No supported image files found in: {folder.resolve()}")

    image_arrays = np.stack([preprocess_image(path) for path in image_paths]).astype(np.float32)
    positive_probabilities = model.predict(image_arrays, verbose=0).reshape(-1)
    negative_probabilities = 1.0 - positive_probabilities

    predictions = []
    for image_path, pos_prob, neg_prob in zip(image_paths, positive_probabilities, negative_probabilities):
        fuzzy_label, fuzzy_score = fuzzy_label_from_probabilities(pos_prob, neg_prob)
        predictions.append(
            {
                "filename": image_path.name,
                "positive_probability": float(pos_prob),
                "negative_probability": float(neg_prob),
                "fis_label": fuzzy_label,
                "fis_score": fuzzy_score,
            }
        )

    return pd.DataFrame(predictions)


# Example usage after training:
# reloaded_model = tf.keras.models.load_model(MODEL_SAVE_PATH)
# predict_folder_with_fis(DATASET_DIR, reloaded_model).head()


## 9. Summary

This notebook now contains a full end-to-end workflow:

1. load and preprocess labeled images,
2. build a tunable CNN,
3. optimize the CNN with a GA using a 5-epoch fitness evaluation, and
4. save the trained model and chosen hyperparameters for later reuse, and
5. refine the final prediction with a Fuzzy Inference System that can explicitly return an uncertain class.

Once `../dataset` contains enough `*_neg.*` and `*_pos.*` images, you can run the notebook top-to-bottom.
